### 1.Ingestion Layer

In [35]:
import pandas as pd
df_author_book = pd.read_csv('Dataset/Author_Book.csv')
df_author = pd.read_csv('Dataset/Author.csv')
df_bookorder = pd.read_csv('Dataset/Book_Order.csv')
df_book = pd.read_csv('Dataset/Book.csv')
df_category = pd.read_csv('Dataset/Category.csv')
df_cst = pd.read_csv('Dataset/Customer.csv')
df_order = pd.read_csv('Dataset/Ordering.csv')


In [36]:
df_author.shape #Test

(100, 2)

In [37]:
df_order

,OrderID,BookID,Price,Quantity
0,1,198,180.70,1.0
1,1,256,114.14,2.0
2,1,393,188.30,1.0
3,1,492,218.49,1.0
4,1,996,NaN,1.0
...,...,...,...,...
32029,7999,910,142.16,2.0
32030,7999,951,81.40,NaN
32031,8000,49,237.53,2.0
32032,8000,217,78.51,1.0


### 2.Bronze Layer

In [38]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [39]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

params = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=tcp:127.0.0.1,1434;"
    "DATABASE=books_dwh;"
    "UID=sa;"
    "PWD=Moaaz2006@;"
    "Encrypt=no;"
    "TrustServerCertificate=yes;"
    "Connection Timeout=5;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

with engine.connect() as conn:
    print("connected")

connected


In [40]:
df_author.to_sql('author', con=engine, schema='bronze', if_exists='append', index=False)
df_author_book.to_sql('author_book', con=engine, schema='bronze', if_exists='append', index=False)
df_book.to_sql('book', con=engine, schema='bronze', if_exists='append', index=False)
df_bookorder.to_sql('book_order', con=engine, schema='bronze', if_exists='append', index=False)
df_category.to_sql('category', con=engine, schema='bronze', if_exists='append', index=False)
df_cst.to_sql('customer', con=engine, schema='bronze', if_exists='append', index=False)
df_order.to_sql('ordering', con=engine, schema='bronze', if_exists='append', index=False)

70

### 3.Silver Layer

In [41]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

params = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=tcp:127.0.0.1,1434;"
    "DATABASE=books_dwh;"
    "UID=sa;"
    "PWD=Moaaz2006@;"
    "Encrypt=no;"
    "TrustServerCertificate=yes;"
    "Connection Timeout=5;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

with engine.connect() as conn:
    print("connected")

connected


In [42]:
df_author_book = pd.read_sql('SELECT * FROM bronze.author_book', con=engine)
df_author = pd.read_sql('SELECT * FROM bronze.author', con=engine)
df_bookorder = pd.read_sql('SELECT * FROM bronze.book_order', con=engine)
df_book = pd.read_sql('SELECT * FROM bronze.book', con=engine)
df_category = pd.read_sql('SELECT * FROM bronze.category', con=engine)
df_cst = pd.read_sql('SELECT * FROM bronze.customer', con=engine)
df_order = pd.read_sql('SELECT * FROM bronze.ordering', con=engine)

In [43]:
df_author_book['AuthorID'] = df_author_book['AuthorID'].astype(int)
df_author_book['BookID'] = df_author_book['BookID'].astype(int)
df_author['AuthorID'] = df_author['AuthorID'].astype(int)
df_bookorder['OrderID'] = df_bookorder['OrderID'].astype(int)
df_book['BookID'] = df_book['BookID'].astype(int)
df_category['CategoryID'] = df_category['CategoryID'].astype(int)
df_cst['CustomerID'] = df_cst['CustomerID'].astype(int)
df_order['OrderID'] = df_order['OrderID'].astype(int)


In [44]:
df_bookorder.isnull().sum()

OrderID          0
CustomerID    1396
OrderDate        0
dtype: int64

In [45]:
df_bookorder['CustomerID']=df_bookorder['CustomerID'].fillna('0', inplace=True)
df_bookorder.isnull().sum()

/tmp/ipykernel_174615/3668669431.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_bookorder['CustomerID']=df_bookorder['CustomerID'].fillna('0', inplace=True)


OrderID       0
CustomerID    0
OrderDate     0
dtype: int64

In [46]:
df_cst.dtypes
df_cst.isnull().sum()

CustomerID       0
FirstName     2872
LastName      3336
ZipCode          0
City          3488
State            0
dtype: int64

In [47]:
df_cst['FirstName']=df_cst['FirstName'].fillna(' ', inplace=True)
df_cst['LastName']=df_cst['LastName'].fillna(' ', inplace=True)
df_cst['City']=df_cst['City'].fillna('0', inplace=True)

/tmp/ipykernel_174615/2711785648.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_cst['FirstName']=df_cst['FirstName'].fillna(' ', inplace=True)
/tmp/ipykernel_174615/2711785648.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace 

In [48]:
df_order.dtypes

OrderID     int64
BookID        str
Price         str
Quantity      str
dtype: object

In [49]:
df_author.to_sql('author', con=engine, schema='silver', if_exists='append', index=False)
df_author_book.to_sql('author_book', con=engine, schema='silver', if_exists='append', index=False)
df_book.to_sql('book', con=engine, schema='silver', if_exists='append', index=False)
df_bookorder.to_sql('book_order', con=engine, schema='silver', if_exists='append', index=False)
df_category.to_sql('category', con=engine, schema='silver', if_exists='append', index=False)
df_cst.to_sql('customer', con=engine, schema='silver', if_exists='append', index=False)
df_order.to_sql('ordering', con=engine, schema='silver', if_exists='append', index=False)

280

### 4.Gold Layer

In [50]:
df_author_book = pd.read_sql('SELECT * FROM silver.author_book', con=engine)
df_author = pd.read_sql('SELECT * FROM silver.author', con=engine)
df_bookorder = pd.read_sql('SELECT * FROM silver.book_order', con=engine)
df_book = pd.read_sql('SELECT * FROM silver.book', con=engine)
df_category = pd.read_sql('SELECT * FROM silver.category', con=engine)
df_cst = pd.read_sql('SELECT * FROM silver.customer', con=engine)
df_order = pd.read_sql('SELECT * FROM silver.ordering', con=engine)

In [51]:
df_cst['FullName'] = df_cst['FirstName'] + ' ' + df_cst['LastName']

In [52]:
df_cst.drop(['FirstName', 'LastName'], axis=1, inplace=True)

In [53]:
df_cst.to_sql('dim_customer', con=engine, schema='gold', if_exists='append', index=False)

402

In [54]:
df_output = df_book.merge(df_author_book,  how='left',on='BookID',)

In [55]:
df_output = df_output.merge(df_author,  how='left',on='AuthorID',)

In [56]:
df_output=df_output[['BookID', 'CategoryID','Title',  'Year', 'BookDescription', 'AuthorName']]

In [ ]:
df_output.to_sql('dim_book', con=engine, schema='gold', if_exists='append', index=False)

In [ ]:
df_fact=df_order.merge(df_bookorder,  how='left',on='OrderID',)
df_fact['Quantity']=df_fact['Quantity'].fillna('0', inplace=True)
df_fact['Price']=df_fact['Price'].fillna('0', inplace=True)
df_fact['Quantity'] = df_fact['Quantity'].astype(int)
df_fact['Price'] = df_fact['Price'].astype(float)
df_fact['total_amount']=df_fact['Quantity'] * df_fact['Price']
dim_customer=pd.read_sql('SELECT * FROM gold.dim_customer', con=engine)
dim_book=pd.read_sql('SELECT * FROM gold.dim_customer', con=engine)
df_fact_output=df_fact.merge(dim_customer,  how='left',on='CustomerID',)
df_fact_output=df_fact_output.merge(dim_book,  how='left',on='BookID',)

/tmp/ipykernel_174615/862575232.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_fact['Quantity']=df_fact['Quantity'].fillna('0', inplace=True)
/tmp/ipykernel_174615/862575232.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace me

ValueError: You are trying to merge on str and int64 columns for key 'CustomerID'. If you wish to proceed you should use pd.concat